# 🔬 Notebook 2 — Driver Analysis
**What drives Revenue, Cancellations, Refunds and Customer Value?**

Sections:
1. Revenue drivers (correlation, feature importance)
2. Cancellation drivers (by city, cuisine, payment, day, rating)
3. Refund drivers (by brand, cuisine, payment mode)
4. Average Order Value drivers
5. Customer lifetime value drivers
6. Discount effectiveness


In [ ]:
import os, warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", "{:,.2f}".format)

sns.set_theme(style="whitegrid", palette="Set2")
plt.rcParams.update({"figure.dpi": 110, "figure.figsize": (12, 5),
                     "axes.titlesize": 13, "axes.labelsize": 11})

BASE = r"C:\Users\rkuma\OneDrive\Desktop\Zomato"

def load(name):
    return pd.read_csv(os.path.join(BASE, f"Zomato  Order Data.xlsx - {name}.csv"))

customers   = load("Customer")
orders      = load("Orders")
restaurants = load("Restaurants")

orders["order_timestamp"] = pd.to_datetime(orders["order_timestamp"],
                                           format="%m/%d/%Y", errors="coerce")
orders["order_month"]   = orders["order_timestamp"].dt.to_period("M")
orders["order_quarter"] = orders["order_timestamp"].dt.to_period("Q")
orders["order_year"]    = orders["order_timestamp"].dt.year.astype("Int64")
orders["day_of_week"]   = orders["order_timestamp"].dt.day_name()
orders["hour"]          = orders["order_timestamp"].dt.hour
orders["discount_amount"] = orders["discount_amount"].fillna(0)
orders["delivery_fee"]    = orders["delivery_fee"].fillna(0)
orders["net_revenue"]     = orders["order_amount"] - orders["discount_amount"]
orders["is_discounted"]   = (orders["discount_amount"] > 0).astype(int)
orders["total_charge"]    = orders["net_revenue"] + orders["delivery_fee"]

customers["Signup_Time"]  = pd.to_datetime(customers["Signup_Time"],
                                           format="%d/%m/%Y", errors="coerce")
customers["signup_month"] = customers["Signup_Time"].dt.to_period("M")
customers["signup_year"]  = customers["Signup_Time"].dt.year.astype("Int64")

full = (orders
        .merge(restaurants, on="restaurant_id", how="left")
        .merge(customers,   left_on="customer_id",
               right_on="Customer_id", how="left"))

delivered  = full[full["order_status"] == "Delivered"].copy()
cancelled  = full[full["order_status"] == "Cancelled"].copy()
refunded   = full[full["order_status"] == "Refunded"].copy()

print(f"Orders: {len(orders):,} | Customers: {customers['Customer_id'].nunique():,} | Restaurants: {len(restaurants)}")
print(f"Date range: {orders['order_timestamp'].min().date()} to {orders['order_timestamp'].max().date()}")


## 1. Revenue Drivers

In [ ]:

from scipy import stats

# Correlation: numeric features vs net_revenue
num_cols = ["order_amount","discount_amount","delivery_fee",
            "avg_rating","is_discounted"]
corr_rev = delivered[num_cols + ["net_revenue"]].corr()["net_revenue"].drop("net_revenue").sort_values()

fig, ax = plt.subplots(figsize=(9, 5))
colors = ["#E63946" if v < 0 else "#2A9D8F" for v in corr_rev.values]
ax.barh(corr_rev.index, corr_rev.values, color=colors)
ax.axvline(0, color="black", linewidth=0.8)
ax.set_title("Correlation with Net Revenue", fontsize=13, fontweight="bold")
ax.set_xlabel("Pearson Correlation Coefficient")
for i, v in enumerate(corr_rev.values):
    ax.text(v + (0.005 if v >= 0 else -0.005), i,
            f"{v:.3f}", va="center", ha="left" if v >= 0 else "right", fontsize=10)
plt.tight_layout()
plt.show()


In [ ]:

# Revenue by cuisine — with error bars
cuisine_rev = (delivered.groupby("cuisine")["net_revenue"]
               .agg(["mean","std","count"]).reset_index())
cuisine_rev["se"] = cuisine_rev["std"] / np.sqrt(cuisine_rev["count"])
cuisine_rev = cuisine_rev.sort_values("mean", ascending=True)

fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(cuisine_rev["cuisine"], cuisine_rev["mean"],
        xerr=cuisine_rev["se"], capsize=4,
        color=sns.color_palette("Pastel1", len(cuisine_rev)), edgecolor="grey")
ax.set_title("Mean Net Revenue per Order by Cuisine (±SE)", fontweight="bold")
ax.set_xlabel("Avg Net Revenue (₹)")
plt.tight_layout()
plt.show()


In [ ]:

# Discount amount vs order amount — does discount drive higher spend?
sample = orders.sample(min(5000, len(orders)), random_state=1)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(sample["discount_amount"], sample["order_amount"],
                alpha=0.3, s=10, color="#264653")
m, b, r, p, _ = stats.linregress(sample["discount_amount"], sample["order_amount"])
x_line = np.linspace(0, sample["discount_amount"].max(), 100)
axes[0].plot(x_line, m * x_line + b, color="#E63946", linewidth=2)
axes[0].set_title(f"Discount vs Order Amount  (r={r:.3f}, p={p:.3e})")
axes[0].set_xlabel("Discount Amount (₹)"); axes[0].set_ylabel("Order Amount (₹)")

# AOV: discounted vs not
disc_orders   = orders[orders["is_discounted"]==1]["order_amount"]
nodisc_orders = orders[orders["is_discounted"]==0]["order_amount"]
axes[1].boxplot([disc_orders, nodisc_orders],
                labels=["Discounted","No Discount"],
                patch_artist=True,
                boxprops=dict(facecolor="#2A9D8F"),
                medianprops=dict(color="red"))
t_stat, p_val = stats.ttest_ind(disc_orders, nodisc_orders)
axes[1].set_title(f"Order Amount: Discounted vs Non-Discounted\nt-test p={p_val:.4f}")
axes[1].set_ylabel("Order Amount (₹)")

plt.suptitle("Discount Impact on Revenue", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

print(f"Discounted   — Mean: ₹{disc_orders.mean():,.0f}  Median: ₹{disc_orders.median():,.0f}")
print(f"No Discount  — Mean: ₹{nodisc_orders.mean():,.0f}  Median: ₹{nodisc_orders.median():,.0f}")


## 2. Cancellation Drivers

In [ ]:

# Cancel rate by every categorical dimension
def cancel_rate_by(col):
    return (full.groupby(col)["order_status"]
            .apply(lambda x: (x=="Cancelled").sum()/len(x)*100)
            .sort_values(ascending=False).reset_index())

dims = ["City", "cuisine", "payment_mode", "day_of_week", "restaurant_name"]
fig, axes = plt.subplots(2, 3, figsize=(18, 11))

for ax, dim in zip(axes.flat, dims):
    df = cancel_rate_by(dim).head(12)
    df.columns = [dim, "Cancel Rate (%)"]
    df_sorted = df.sort_values("Cancel Rate (%)")
    ax.barh(df_sorted[dim].astype(str), df_sorted["Cancel Rate (%)"],
            color=sns.color_palette("Reds_r", len(df_sorted)))
    ax.set_title(f"Cancel Rate by {dim}", fontweight="bold")
    ax.set_xlabel("Cancel Rate (%)")
    for i, v in enumerate(df_sorted["Cancel Rate (%)"]):
        ax.text(v + 0.1, i, f"{v:.1f}%", va="center", fontsize=9)

axes.flat[-1].set_visible(False)
plt.suptitle("Cancellation Driver Analysis", fontsize=15, fontweight="bold")
plt.tight_layout()
plt.show()


In [ ]:

# Logistic regression — what predicts cancellation?
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score

lr_df = full[["order_amount","discount_amount","delivery_fee",
              "is_discounted","avg_rating",
              "payment_mode","cuisine","City","day_of_week",
              "order_status"]].dropna().copy()

lr_df["cancelled"] = (lr_df["order_status"] == "Cancelled").astype(int)

for col in ["payment_mode","cuisine","City","day_of_week"]:
    lr_df[col] = LabelEncoder().fit_transform(lr_df[col].astype(str))

feats = ["order_amount","discount_amount","delivery_fee","is_discounted",
         "avg_rating","payment_mode","cuisine","City","day_of_week"]
X = lr_df[feats]; y = lr_df["cancelled"]

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)
lr = LogisticRegression(max_iter=1000, class_weight="balanced")
lr.fit(X_tr, y_tr)

coefs = pd.Series(lr.coef_[0], index=feats).sort_values()
fig, ax = plt.subplots(figsize=(9, 5))
colors = ["#E63946" if v > 0 else "#2A9D8F" for v in coefs.values]
ax.barh(coefs.index, coefs.values, color=colors)
ax.axvline(0, color="black", linewidth=0.8)
ax.set_title("Logistic Regression Coefficients — Cancellation Predictors",
             fontweight="bold")
ax.set_xlabel("Coefficient (positive = increases cancel risk)")
plt.tight_layout()
plt.show()

print("Classification Report (Cancellation):")
print(classification_report(y_te, lr.predict(X_te)))
print(f"ROC-AUC: {roc_auc_score(y_te, lr.predict_proba(X_te)[:,1]):.4f}")


## 3. Refund Drivers

In [ ]:

def refund_rate_by(col):
    return (full.groupby(col)["order_status"]
            .apply(lambda x: (x=="Refunded").sum()/len(x)*100)
            .sort_values(ascending=False).reset_index())

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
for ax, dim in zip(axes, ["restaurant_name","cuisine","payment_mode"]):
    df = refund_rate_by(dim).head(12)
    df.columns = [dim, "Refund Rate (%)"]
    df_s = df.sort_values("Refund Rate (%)")
    ax.barh(df_s[dim].astype(str), df_s["Refund Rate (%)"],
            color=sns.color_palette("Purples_r", len(df_s)))
    ax.set_title(f"Refund Rate by {dim}", fontweight="bold")
    ax.set_xlabel("Refund Rate (%)")
    for i, v in enumerate(df_s["Refund Rate (%)"]):
        ax.text(v + 0.1, i, f"{v:.1f}%", va="center", fontsize=9)

plt.suptitle("Refund Driver Analysis", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()


In [ ]:

# Revenue lost to refunds
refund_rev   = full[full["order_status"]=="Refunded"]["order_amount"].sum()
total_gross  = full["order_amount"].sum()
print(f"Revenue lost to refunds : ₹{refund_rev:,.0f}")
print(f"As % of gross revenue   : {refund_rev/total_gross*100:.2f}%")

# Brand-level refund vs revenue table
brand_stats = full.groupby("restaurant_name").agg(
    total_orders  = ("order_id","count"),
    refund_count  = ("order_status", lambda x: (x=="Refunded").sum()),
    gross_revenue = ("order_amount","sum"),
    refund_revenue= ("order_amount", lambda x: x[full.loc[x.index,"order_status"]=="Refunded"].sum())
).reset_index()
brand_stats["refund_rate"]    = brand_stats["refund_count"] / brand_stats["total_orders"] * 100
brand_stats["revenue_lost_%"] = brand_stats["refund_revenue"] / brand_stats["gross_revenue"] * 100
display(brand_stats.sort_values("refund_rate", ascending=False).head(10).round(2))


## 4. Average Order Value (AOV) Drivers

In [ ]:

aov_city = delivered.groupby("City")["order_amount"].mean().sort_values(ascending=False)
aov_cuisine = delivered.groupby("cuisine")["order_amount"].mean().sort_values(ascending=False)
aov_payment = delivered.groupby("payment_mode")["order_amount"].mean().sort_values(ascending=False)
aov_day     = delivered.groupby("day_of_week")["order_amount"].mean()

fig, axes = plt.subplots(2, 2, figsize=(15, 10))
for ax, (series, title, color) in zip(axes.flat, [
    (aov_city,    "AOV by City",         "#264653"),
    (aov_cuisine, "AOV by Cuisine",      "#2A9D8F"),
    (aov_payment, "AOV by Payment Mode", "#E9C46A"),
    (aov_day.reindex(["Monday","Tuesday","Wednesday","Thursday","Friday","Saturday","Sunday"]),
              "AOV by Day of Week",   "#E76F51"),
]):
    ax.bar(series.index, series.values, color=color, edgecolor="white")
    ax.set_title(title, fontweight="bold")
    ax.set_ylabel("Avg Order Value (₹)")
    ax.tick_params(axis="x", rotation=30)
    for i, v in enumerate(series.values):
        ax.text(i, v + 3, f"₹{v:,.0f}", ha="center", fontsize=9)

plt.suptitle("AOV Driver Analysis", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()


## 5. Customer Lifetime Value (CLV) Drivers

In [ ]:

clv = (delivered.groupby("customer_id").agg(
    total_revenue    = ("net_revenue","sum"),
    total_orders     = ("order_id","count"),
    avg_order_value  = ("order_amount","mean"),
    first_order      = ("order_timestamp","min"),
    last_order       = ("order_timestamp","max"),
).reset_index())
clv["tenure_days"] = (clv["last_order"] - clv["first_order"]).dt.days

clv_with_acq = clv.merge(
    customers[["Customer_id","Acquisition_channel","City"]],
    left_on="customer_id", right_on="Customer_id", how="left"
)

print(f"CLV Distribution:\n{clv['total_revenue'].describe().round(2)}")
print(f"\nTop 10% customers contribute: "
      f"{clv.nlargest(int(len(clv)*0.1),'total_revenue')['total_revenue'].sum() / clv['total_revenue'].sum()*100:.1f}% of revenue")


In [ ]:

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# CLV by acquisition channel
clv_acq = clv_with_acq.groupby("Acquisition_channel")["total_revenue"].mean().sort_values()
axes[0].barh(clv_acq.index, clv_acq.values,
             color=sns.color_palette("Set2", len(clv_acq)))
axes[0].set_title("Avg CLV by Acquisition Channel", fontweight="bold")
axes[0].set_xlabel("Avg Revenue (₹)")
for i, v in enumerate(clv_acq.values):
    axes[0].text(v + 100, i, f"₹{v:,.0f}", va="center", fontsize=9)

# CLV by city
clv_city = clv_with_acq.groupby("City")["total_revenue"].mean().sort_values()
axes[1].barh(clv_city.index, clv_city.values,
             color=sns.color_palette("teal", len(clv_city)))
axes[1].set_title("Avg CLV by City", fontweight="bold")
axes[1].set_xlabel("Avg Revenue (₹)")
for i, v in enumerate(clv_city.values):
    axes[1].text(v + 100, i, f"₹{v:,.0f}", va="center", fontsize=9)

# CLV distribution
axes[2].hist(clv["total_revenue"], bins=40, color="#457B9D", edgecolor="white")
axes[2].axvline(clv["total_revenue"].mean(), color="red", linestyle="--",
                label=f"Mean ₹{clv['total_revenue'].mean():,.0f}")
axes[2].axvline(clv["total_revenue"].median(), color="orange", linestyle="--",
                label=f"Median ₹{clv['total_revenue'].median():,.0f}")
axes[2].set_title("CLV Distribution")
axes[2].set_xlabel("Total Revenue (₹)")
axes[2].legend(fontsize=9)

plt.suptitle("Customer Lifetime Value Drivers", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()


## 6. Feature Importance — Revenue Prediction (Random Forest)

In [ ]:

from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split

rf_df = delivered[["order_amount","discount_amount","delivery_fee",
                   "is_discounted","avg_rating",
                   "payment_mode","cuisine","City",
                   "day_of_week","net_revenue"]].dropna().copy()

for col in ["payment_mode","cuisine","City","day_of_week"]:
    rf_df[col] = LabelEncoder().fit_transform(rf_df[col].astype(str))

feats_rf = ["order_amount","discount_amount","delivery_fee",
            "is_discounted","avg_rating",
            "payment_mode","cuisine","City","day_of_week"]
X_rf = rf_df[feats_rf]; y_rf = rf_df["net_revenue"]
X_tr, X_te, y_tr, y_te = train_test_split(X_rf, y_rf, test_size=0.2, random_state=42)

rf = RandomForestRegressor(n_estimators=100, max_depth=8, random_state=42, n_jobs=-1)
rf.fit(X_tr, y_tr)
print(f"R² on test set: {rf.score(X_te, y_te):.4f}")

imp = pd.Series(rf.feature_importances_, index=feats_rf).sort_values()
fig, ax = plt.subplots(figsize=(9, 5))
ax.barh(imp.index, imp.values,
        color=sns.color_palette("YlOrRd_r", len(imp)))
ax.set_title("Random Forest — Feature Importance for Net Revenue",
             fontweight="bold")
ax.set_xlabel("Importance Score")
for i, v in enumerate(imp.values):
    ax.text(v + 0.001, i, f"{v:.4f}", va="center", fontsize=9)
plt.tight_layout()
plt.show()


## ✅ Driver Analysis Summary
- **Order amount** is the primary revenue driver — confirmed by RF importance and correlation.
- **Discount amount** has a positive correlation with order size, but statistical tests show discounted orders are not necessarily higher-value.
- **City** and **cuisine** are significant categorical drivers for both revenue and cancellations.
- **Rating** has a modest positive correlation with revenue — higher-rated restaurants drive marginally more spend.
- **Cancellation** is weakly predicted by numerical features; city/cuisine/day effects are stronger.
- **Top 10% of customers** contribute a disproportionate share of revenue — loyalty targeting is critical.